# 🔬 MicroSR — Physics-Constrained Microscopy Super-Resolution

> A diffusion model that enhances blurry microscope images with PSF physics constraint.
>
> **Run this notebook in Colab (Runtime → Change runtime type → T4 GPU)**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)]()

---

## ⚙️ Setup

Clone the repository and install dependencies.

In [ ]:
import os, sys, torch
print(f'Python: {sys.version[:5]} | PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}')

In [ ]:
# Clone repo
!git clone https://github.com/Tharungowdapr/MicroSR-.git /content/MicroSR
%cd /content/MicroSR

# Install dependencies
!pip install -q tifffile wandb mlflow scikit-image matplotlib gradio tqdm 2>&1 | tail -1
print('Dependencies installed')

In [ ]:
import sys
sys.path.insert(0, '/content/MicroSR')
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from src.model import UNet, DDPM
from src.losses import make_gaussian_psf
print('Imports OK')

## 📦 Generate Synthetic Data

Since BioSR download from Figshare may be blocked in Colab, we generate synthetic microscopy data.

In [ ]:
%cd /content/MicroSR
!python scripts/generate_synthetic_data.py --out_dir data/BioSR --n_train 200 --n_val 20 2>&1 | tail -5

## 🏋️ Train Model

Training with T4 GPU. Reduced settings for Colab speed: 50 epochs.

In [ ]:
!python train.py \
  --data_root data/BioSR \
  --run_dir runs/colab \
  --epochs 50 \
  --batch_size 4 \
  --T 200 \
  --max_train 200 \
  --max_val 20 \
  --device cuda \
  --amp 2>&1 | tail -30

## 🔬 Run Inference on Sample

In [ ]:
# Load trained model
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = UNet(in_ch=2, out_ch=1, base_ch=64).to(device)
ckpt = torch.load('runs/colab/best_model.pt', map_location=device)
model.load_state_dict(ckpt['model'])
model.eval()
ddpm = DDPM(T=200, device=device)
psf = make_gaussian_psf(2.0, device=device)
print(f'Model loaded — epoch {ckpt.get("epoch", "?")}')

In [ ]:
# Load a test image
import tifffile
lr_path = list(Path('data/BioSR').rglob('LR/*.tif'))[0]
img = tifffile.imread(str(lr_path)).astype(np.float32)
img = (img - img.min()) / (img.max() - img.min() + 1e-8)
lr_t = torch.from_numpy(img)[None, None].to(device) * 2 - 1  # [-1,1]

# Generate SR
with torch.no_grad():
    hr_gen = ddpm.sample(model, lr_t, shape=(1, 1, 128, 128))

# Display
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
lr_disp = ((lr_t.squeeze().cpu().clamp(-1,1)+1)/2).numpy()
hr_disp = ((hr_gen.squeeze().cpu().clamp(-1,1)+1)/2).numpy()
ax1.imshow(lr_disp, cmap='gray'); ax1.set_title('LR Input (64x64)')
ax2.imshow(hr_disp, cmap='gray'); ax2.set_title('HR Output (128x128)')
plt.tight_layout(); plt.show()

## 💾 Save Model to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp -r /content/MicroSR/runs/colab /content/drive/MyDrive/microsr_results/
print('Model saved to Google Drive')